# Chapter 22: Four-Dimensional Geometry

Source span: printed pages 396-412; PDF pages 414-430. The PDF is image-only, so this notebook uses the course source map in `Introduction-to-Geometry/AGENTS.md`, the local chapter index, and the existing chapter-22 artifact map for orientation. The prose, code, diagrams, and checks here are original and do not copy textbook text, figures, page crops, or exercise statements.

Chapter goal: make four-dimensional geometry inspectable without pretending that a 2D screen can show all of it. We will use coordinates in `R^4`, perspective and orthographic projections, 3D sections, Schlafli-symbol inequalities, lattice packing counts, and statistical cell summaries. The recurring question is simple: which claim survives projection, slicing, sampling, or a change from a regular cell to a random one?


## Computational translation guide

A point in four-dimensional geometry becomes a length-4 vector. A segment, face, cell, or honeycomb entry becomes an incidence rule on such vectors. A projection is a map that deliberately discards information; it can preserve incidence while damaging lengths and angles. A section is the opposite habit: fix one coordinate or hyperplane and inspect the lower-dimensional object that remains. A regular polytope is summarized by its Schlafli symbol and its `f`-vector `(V, E, F, C)`. A regular honeycomb candidate is tested by local finite-cell conditions plus the Schlafli trigonometric inequality. Equal-sphere packing becomes a lattice, a minimum distance, a kissing number, and a density. Statistical honeycombs replace a single symbol by distributions of cell areas, side counts, and nearest-neighbor scales.

Library routing for this chapter: Plotly handles the interactive 3D views produced by 4D projections and sections; Matplotlib writes durable SVG diagrams and tables; SciPy supplies Voronoi and distance computations for statistical honeycombs; SymPy checks the exact Euclidean boundary case for a honeycomb symbol; Pandas/JSON store auditable tables and invariants.


## Refreshed visual storyboard

| Story item | Representation | Artifact | Learner inspection target | Check |
| --- | --- | --- | --- | --- |
| Tesseract shadow | 4D perspective projection to 3D plus SVG camera view | `schlegel-tesseract-projection.svg`, `tesseract-projection.html` | near and far cubic facets remain incident but not congruent in the drawing | 16 vertices, 32 edges, near facet appears larger |
| 4D section | slices of the 16-cell at several `w` levels | `cross-polytope-section-sweep.svg`, `cross-polytope-section-sweep.html` | sections are octahedra whose volume scales cubically | section volume equals `4/3*(1-|w|)^3` |
| Regular polytopes | f-vector and duality table | `regular-polytope-f-vectors.svg` | dual pairs swap vertices and cells; all boundaries have Euler characteristic 0 | `V - E + F - C = 0` |
| Honeycomb symbols | local admissibility and spherical/Euclidean/hyperbolic classifier | `honeycomb-symbol-admissibility.svg` | `{4,3,4}` sits exactly on the Euclidean boundary | exact SymPy equality and numeric margins |
| Equal 4-sphere packing | D4 root shell around one sphere | `d4-close-packing-neighbors.svg` | 24 neighbor centers touch the central 4-sphere | kissing number 24 and density `pi^2/16` |
| Statistical honeycomb | deterministic random Voronoi proxy and cell statistics | `statistical-honeycomb-voronoi.svg` | a honeycomb can be summarized by distributions, not only one regular symbol | positive cell areas and sensible side-count mean |

Read the sequence as a set of controlled losses. The tesseract view throws away one coordinate, so it is useful only after we agree that incidence is the quantity to protect. The section view fixes the hidden coordinate instead, so it turns a 4D body into an ordinary 3D polyhedron with a measurable volume. The regular-polytope table is even more compressed: it cannot show the cells, but it can catch inconsistent counts and duality mistakes. The honeycomb-symbol panel then asks which symbolic triples pass local tests and which side of the trigonometric boundary they occupy. The packing and statistical-honeycomb sections deliberately move away from exact drawings. A packing is certified by distances and density; a statistical honeycomb is certified by distributions. This is the working discipline for the chapter: every visual names the geometric information it keeps, the information it discards, and the invariant that prevents the picture from becoming decorative.


In [ ]:
from __future__ import annotations

from itertools import combinations, product
from math import cos, pi, sin, sqrt
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.spatial import Voronoi, distance_matrix
import sympy as sp
from IPython.display import Markdown, display

CHAPTER_NO = 22
HERE = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [HERE, *HERE.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not locate the Introduction-to-Geometry book root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, ensure_chapter_artifact_dirs, write_csv, write_json
from utils.course import chapter_by_no

chapter = chapter_by_no(CHAPTER_NO)
ARTIFACT_DIRS = ensure_chapter_artifact_dirs(CHAPTER_NO, BOOK_ROOT)
FIG = ARTIFACT_DIRS["figures"]
HTML = ARTIFACT_DIRS["html"]
CHECKS = ARTIFACT_DIRS["checks"]
TABLES = ARTIFACT_DIRS["tables"]
DATA = ARTIFACT_DIRS["data"]

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

artifact_records = []
check_payload = {
    "chapter": CHAPTER_NO,
    "title": chapter["title"],
    "source_span": {"printed_pages": chapter["printed"], "pdf_pages": chapter["pdf"]},
    "invariants": {},
    "figures": [],
    "html": [],
    "tables": [],
    "data": [],
}


def rel(path: Path) -> str:
    return path.relative_to(BOOK_ROOT).as_posix()


def remember(path: Path, role: str, concept: str, library: str, inspection_target: str) -> Path:
    suffix = path.suffix.lower()
    if suffix == ".svg":
        check_payload["figures"].append(rel(path))
    elif suffix == ".html":
        check_payload["html"].append(rel(path))
    elif suffix == ".csv":
        if "tables" in path.parts:
            check_payload["tables"].append(rel(path))
        else:
            check_payload["data"].append(rel(path))
    elif suffix == ".json":
        check_payload.setdefault("checks", []).append(rel(path))
    artifact_records.append({
        "artifact": rel(path),
        "role": role,
        "concept": concept,
        "library": library,
        "inspection_target": inspection_target,
    })
    return path


palette = {
    "blue": "#2a6fbb",
    "orange": "#d97706",
    "green": "#15803d",
    "red": "#b91c1c",
    "purple": "#6d28d9",
    "gray": "#4b5563",
    "light": "#f8fafc",
}

display(Markdown(f"Artifacts will be written under `{rel(BOOK_ROOT / 'artifacts' / 'chapter-22')}`."))


## 1. Projection: a tesseract as a controlled shadow

A tesseract has 16 vertices and 32 edges in `R^4`. The Schlegel-style view below projects from a point on the fourth-coordinate side onto a 3D hyperplane, then uses a 2D camera only for the static SVG. The drawing is not a metric model: the near cubic facet is intentionally larger than the far cubic facet. What the projection should preserve is incidence: which vertices are connected to which edges.


In [ ]:
def tesseract_vertices_edges() -> tuple[np.ndarray, list[tuple[int, int]]]:
    vertices = np.array(list(product([-1.0, 1.0], repeat=4)))
    edges = []
    for i, j in combinations(range(len(vertices)), 2):
        if np.count_nonzero(vertices[i] != vertices[j]) == 1:
            edges.append((i, j))
    return vertices, edges


def perspective_4d_to_3d(points: np.ndarray, eye_w: float = 2.8) -> np.ndarray:
    denom = eye_w - points[:, 3]
    if np.any(denom <= 0):
        raise ValueError("projection eye must be beyond all fourth coordinates")
    scale = eye_w / denom
    return points[:, :3] * scale[:, None]


def camera_3d_to_2d(points: np.ndarray) -> np.ndarray:
    camera = np.array([
        [0.92, -0.20, 0.25],
        [0.22, 0.56, 0.80],
    ])
    xy = points @ camera.T
    xy -= xy.mean(axis=0)
    xy /= np.abs(xy).max()
    return xy


vertices4, tesseract_edges = tesseract_vertices_edges()
projected3 = perspective_4d_to_3d(vertices4, eye_w=2.8)
projected2 = camera_3d_to_2d(projected3)

near_edges = [edge for edge in tesseract_edges if vertices4[edge[0], 3] == 1 and vertices4[edge[1], 3] == 1]
far_edges = [edge for edge in tesseract_edges if vertices4[edge[0], 3] == -1 and vertices4[edge[1], 3] == -1]
connector_edges = [edge for edge in tesseract_edges if edge not in near_edges and edge not in far_edges]


def mean_projected_length(edges: list[tuple[int, int]]) -> float:
    return float(np.mean([np.linalg.norm(projected3[i] - projected3[j]) for i, j in edges]))


near_mean = mean_projected_length(near_edges)
far_mean = mean_projected_length(far_edges)
assert len(vertices4) == 16
assert len(tesseract_edges) == 32
assert len(near_edges) == 12 and len(far_edges) == 12 and len(connector_edges) == 8
assert near_mean > far_mean

fig, ax = plt.subplots(figsize=(7.6, 5.2))
ax.set_facecolor("#fffdf7")
for edges, color, width, label in [
    (far_edges, palette["blue"], 1.4, "far cubic facet"),
    (connector_edges, palette["gray"], 1.0, "fourth-direction connectors"),
    (near_edges, palette["orange"], 2.2, "near cubic facet"),
]:
    segments = [[projected2[i], projected2[j]] for i, j in edges]
    ax.add_collection(LineCollection(segments, colors=color, linewidths=width, alpha=0.92, label=label))
ax.scatter(projected2[:, 0], projected2[:, 1], s=28, color=np.where(vertices4[:, 3] > 0, palette["orange"], palette["blue"]), zorder=3)
for idx, xy in enumerate(projected2):
    if vertices4[idx, 3] > 0:
        ax.text(xy[0] + 0.02, xy[1] + 0.02, f"{idx}", fontsize=7, color=palette["orange"])
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
ax.set_title("Schlegel-style tesseract projection: incidence survives, metric scale does not")
ax.legend(loc="lower left", frameon=False)
projection_svg = FIG / "schlegel-tesseract-projection.svg"
fig.savefig(projection_svg, format="svg")
plt.close(fig)
remember(projection_svg, "figure", "Schlegel-style tesseract projection", "Matplotlib SVG", "Compare near and far cubic facets and trace the 8 connector edges")

line_x, line_y, line_z = [], [], []
for i, j in tesseract_edges:
    line_x += [projected3[i, 0], projected3[j, 0], None]
    line_y += [projected3[i, 1], projected3[j, 1], None]
    line_z += [projected3[i, 2], projected3[j, 2], None]
fig3d = go.Figure()
fig3d.add_trace(go.Scatter3d(x=line_x, y=line_y, z=line_z, mode="lines", line=dict(color="#4b5563", width=5), name="tesseract edges"))
fig3d.add_trace(go.Scatter3d(
    x=projected3[:, 0], y=projected3[:, 1], z=projected3[:, 2],
    mode="markers+text",
    marker=dict(size=5, color=vertices4[:, 3], colorscale=[[0, "#2a6fbb"], [1, "#d97706"]], colorbar=dict(title="original w")),
    text=[str(i) for i in range(len(vertices4))], textposition="top center", name="vertices"))
fig3d.update_layout(
    title="Interactive 3D shadow of a 4D tesseract",
    scene=dict(aspectmode="data", xaxis_title="projected x", yaxis_title="projected y", zaxis_title="projected z"),
    margin=dict(l=0, r=0, t=40, b=0),
)
projection_html = HTML / "tesseract-projection.html"
fig3d.write_html(projection_html, include_plotlyjs=True, full_html=True)
remember(projection_html, "interactive", "Tesseract 4D-to-3D projection", "Plotly HTML", "Rotate the shadow and notice that incidence, not length, is the reliable datum")

check_payload["invariants"]["tesseract_projection"] = {
    "vertices": int(len(vertices4)),
    "edges": int(len(tesseract_edges)),
    "near_facet_edges": int(len(near_edges)),
    "far_facet_edges": int(len(far_edges)),
    "connector_edges": int(len(connector_edges)),
    "near_projected_edge_mean": near_mean,
    "far_projected_edge_mean": far_mean,
    "near_to_far_projected_edge_ratio": near_mean / far_mean,
}

display_artifact(projection_svg, width=760)
display(Markdown(f"Open the interactive projection: [{projection_html.name}]({rel(projection_html)})."))
check_payload["invariants"]["tesseract_projection"]


## 2. Section: slicing the 16-cell

The 16-cell is the 4D cross-polytope. In coordinates it is the unit ball of the `l1` norm: `|x| + |y| + |z| + |w| <= 1`. If we fix `w = t`, the remaining section is the 3D octahedron `|x| + |y| + |z| <= 1 - |t|`. This makes slicing measurable: as the cutting plane moves toward a vertex, the octahedron shrinks and its volume scales as a cube.


In [ ]:
def octahedron_vertices(scale: float) -> np.ndarray:
    return np.array([
        [scale, 0, 0], [-scale, 0, 0], [0, scale, 0],
        [0, -scale, 0], [0, 0, scale], [0, 0, -scale],
    ], dtype=float)


octa_faces = np.array([
    [0, 2, 4], [2, 1, 4], [1, 3, 4], [3, 0, 4],
    [2, 0, 5], [1, 2, 5], [3, 1, 5], [0, 3, 5],
])


def octahedron_volume(scale: float) -> float:
    return 4.0 * scale**3 / 3.0


section_levels = [-0.70, 0.0, 0.70]
section_rows = []
fig = plt.figure(figsize=(8.8, 4.8))
ax = fig.add_subplot(111, projection="3d")
colors = ["#2a6fbb", "#15803d", "#d97706"]
for offset_index, (t, color) in enumerate(zip(section_levels, colors)):
    scale = 1.0 - abs(t)
    verts = octahedron_vertices(scale)
    shifted = verts + np.array([2.8 * (offset_index - 1), 0, 0])
    poly = Poly3DCollection([shifted[face] for face in octa_faces], facecolor=color, edgecolor="#111827", alpha=0.25, linewidth=0.8)
    ax.add_collection3d(poly)
    ax.scatter(shifted[:, 0], shifted[:, 1], shifted[:, 2], color=color, s=18)
    ax.text(2.8 * (offset_index - 1), -1.15, -1.05, f"w = {t:+.2f}\nvolume = {octahedron_volume(scale):.3f}", color=color, ha="center")
    section_rows.append({"w_level": t, "section_scale": scale, "octahedron_volume": octahedron_volume(scale)})
ax.set_title("Sections of the 4D cross-polytope: offset copies of the resulting 3D octahedra")
ax.set_xlabel("display x")
ax.set_ylabel("section y")
ax.set_zlabel("section z")
ax.set_box_aspect((3.8, 1.4, 1.4))
ax.view_init(elev=22, azim=-52)
section_svg = FIG / "cross-polytope-section-sweep.svg"
fig.savefig(section_svg, format="svg")
plt.close(fig)
remember(section_svg, "figure", "16-cell section sweep", "Matplotlib mplot3d SVG", "Compare how octahedral sections shrink as |w| increases")

fig_section = go.Figure()
for t, color in zip(section_levels, colors):
    scale = 1.0 - abs(t)
    verts = octahedron_vertices(scale)
    offset = np.array([3.0 * t, 0, 0])
    shifted = verts + offset
    fig_section.add_trace(go.Mesh3d(
        x=shifted[:, 0], y=shifted[:, 1], z=shifted[:, 2],
        i=octa_faces[:, 0], j=octa_faces[:, 1], k=octa_faces[:, 2],
        color=color, opacity=0.35, name=f"w = {t:+.2f}"))
    fig_section.add_trace(go.Scatter3d(
        x=shifted[:, 0], y=shifted[:, 1], z=shifted[:, 2], mode="markers",
        marker=dict(size=4, color=color), showlegend=False))
fig_section.update_layout(
    title="Interactive cross-polytope sections; x-offset separates the displayed slices",
    scene=dict(aspectmode="data", xaxis_title="display x", yaxis_title="section y", zaxis_title="section z"),
    margin=dict(l=0, r=0, t=40, b=0),
)
section_html = HTML / "cross-polytope-section-sweep.html"
fig_section.write_html(section_html, include_plotlyjs=True, full_html=True)
remember(section_html, "interactive", "16-cell section sweep", "Plotly HTML", "Rotate the octahedral sections and compare their shared combinatorics")

section_csv = DATA / "cross-polytope-section-volumes.csv"
write_csv(section_csv, section_rows)
remember(section_csv, "data", "Section volume scale", "Pandas/CSV", "Check the cubic volume law for sections of the 16-cell")

for row in section_rows:
    expected = 4.0 * row["section_scale"]**3 / 3.0
    assert abs(row["octahedron_volume"] - expected) < 1e-12
assert section_rows[1]["octahedron_volume"] > section_rows[0]["octahedron_volume"]
check_payload["invariants"]["cross_polytope_sections"] = {
    "levels": section_levels,
    "volumes": [row["octahedron_volume"] for row in section_rows],
    "formula": "volume(w) = 4/3 * (1 - abs(w))^3 for abs(w) <= 1",
}

display_artifact(section_svg, width=760)
display(Markdown(f"Open the interactive section view: [{section_html.name}]({rel(section_html)})."))
pd.DataFrame(section_rows)


## 3. Regular 4-polytopes: counts, duality, and the boundary Euler check

A 4-polytope has vertices, edges, polygonal faces, and polyhedral cells. The boundary is a 3-dimensional closed complex, so its Euler characteristic is zero: `V - E + F - C = 0`. This table uses the six convex regular 4-polytopes. The check is intentionally modest but powerful: a wrong count usually breaks Euler, and a wrong dual pairing fails by not swapping vertices and cells.


In [ ]:
regular_polytopes = [
    {"name": "5-cell", "symbol": "{3,3,3}", "V": 5, "E": 10, "F": 10, "C": 5, "dual": "5-cell"},
    {"name": "tesseract", "symbol": "{4,3,3}", "V": 16, "E": 32, "F": 24, "C": 8, "dual": "16-cell"},
    {"name": "16-cell", "symbol": "{3,3,4}", "V": 8, "E": 24, "F": 32, "C": 16, "dual": "tesseract"},
    {"name": "24-cell", "symbol": "{3,4,3}", "V": 24, "E": 96, "F": 96, "C": 24, "dual": "24-cell"},
    {"name": "120-cell", "symbol": "{5,3,3}", "V": 600, "E": 1200, "F": 720, "C": 120, "dual": "600-cell"},
    {"name": "600-cell", "symbol": "{3,3,5}", "V": 120, "E": 720, "F": 1200, "C": 600, "dual": "120-cell"},
]
poly_df = pd.DataFrame(regular_polytopes)
poly_df["euler_boundary"] = poly_df["V"] - poly_df["E"] + poly_df["F"] - poly_df["C"]
assert (poly_df["euler_boundary"] == 0).all()
for _, row in poly_df.iterrows():
    dual = poly_df.loc[poly_df["name"] == row["dual"]].iloc[0]
    assert row["V"] == dual["C"] and row["C"] == dual["V"]
    assert row["E"] == dual["F"] and row["F"] == dual["E"]

poly_csv = TABLES / "regular-polytope-f-vectors.csv"
write_csv(poly_csv, poly_df.to_dict("records"))
remember(poly_csv, "table", "Regular 4-polytope f-vectors", "Pandas/CSV", "Audit the vertex-edge-face-cell counts and duality pairs")

fig, (ax_table, ax_graph) = plt.subplots(1, 2, figsize=(11.4, 5.0), gridspec_kw={"width_ratios": [2.8, 1.2]})
ax_table.axis("off")
show_cols = ["name", "symbol", "V", "E", "F", "C", "dual", "euler_boundary"]
table = ax_table.table(cellText=poly_df[show_cols].values, colLabels=show_cols, cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1.0, 1.45)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor("#e0f2fe")
        cell.set_text_props(weight="bold")
    elif col == show_cols.index("euler_boundary"):
        cell.set_facecolor("#dcfce7")

G = nx.Graph()
for item in regular_polytopes:
    G.add_node(item["name"])
    G.add_edge(item["name"], item["dual"])
pos = {
    "5-cell": (0, 1.2), "tesseract": (-1.2, 0.35), "16-cell": (1.2, 0.35),
    "24-cell": (0, -0.55), "120-cell": (-1.25, -1.35), "600-cell": (1.25, -1.35),
}
nx.draw_networkx_edges(G, pos, ax=ax_graph, edge_color="#64748b", width=1.6)
nx.draw_networkx_nodes(G, pos, ax=ax_graph, node_color="#fef3c7", edgecolors="#92400e", node_size=1250)
nx.draw_networkx_labels(G, pos, ax=ax_graph, font_size=8)
ax_graph.set_title("Duality swaps V/C and E/F")
ax_graph.axis("off")
fig.suptitle("The six convex regular 4-polytopes: f-vectors and duality checks")
poly_svg = FIG / "regular-polytope-f-vectors.svg"
fig.savefig(poly_svg, format="svg")
plt.close(fig)
remember(poly_svg, "figure", "Regular 4-polytope f-vectors and duality", "Matplotlib + NetworkX SVG", "Check Euler zero and read the duality graph")

check_payload["invariants"]["regular_polytopes"] = {
    "polytope_count": int(len(poly_df)),
    "all_euler_boundary_zero": bool((poly_df["euler_boundary"] == 0).all()),
    "self_dual": poly_df.loc[poly_df["name"] == poly_df["dual"], "name"].tolist(),
}

display_artifact(poly_svg, width=900)
poly_df


## 4. Honeycomb-symbol admissibility

For a regular object with symbol `{p,q,r}`, the facets have type `{p,q}` and the vertex figure has type `{q,r}`. As a local check, both must be finite regular polyhedra, so `(p-2)(q-2) < 4` and `(q-2)(r-2) < 4`. The trigonometric margin

`sin(pi/p) sin(pi/r) - cos(pi/q)`

then separates spherical finite regular 4-polytopes, the Euclidean honeycomb boundary, and hyperbolic candidates. The exact equality `{4,3,4}` is the cubic honeycomb boundary case.


In [ ]:
def local_polyhedron_ok(a: int, b: int) -> bool:
    return (a - 2) * (b - 2) < 4


def symbol_margin(p: int, q: int, r: int) -> float:
    return sin(pi / p) * sin(pi / r) - cos(pi / q)


def classify_symbol(p: int, q: int, r: int, tol: float = 1e-10) -> str:
    if not (local_polyhedron_ok(p, q) and local_polyhedron_ok(q, r)):
        return "local-invalid"
    margin = symbol_margin(p, q, r)
    if margin > tol:
        return "spherical-finite"
    if abs(margin) <= tol:
        return "euclidean-honeycomb"
    return "hyperbolic-candidate"


rows = []
for p in range(3, 9):
    for q in range(3, 6):
        for r in range(3, 9):
            rows.append({
                "p": p, "q": q, "r": r,
                "symbol": f"{{{p},{q},{r}}}",
                "facet_ok": local_polyhedron_ok(p, q),
                "vertex_figure_ok": local_polyhedron_ok(q, r),
                "margin": symbol_margin(p, q, r),
                "classification": classify_symbol(p, q, r),
            })
symbol_df = pd.DataFrame(rows)
symbol_csv = TABLES / "honeycomb-symbol-admissibility.csv"
write_csv(symbol_csv, symbol_df.to_dict("records"))
remember(symbol_csv, "table", "Honeycomb symbol admissibility", "Pandas/CSV", "Filter triples by local cell conditions and the Schlafli margin")

exact_euclidean = sp.simplify(sp.sin(sp.pi / 4) * sp.sin(sp.pi / 4) - sp.cos(sp.pi / 3))
exact_spherical = sp.simplify(sp.sin(sp.pi / 3) * sp.sin(sp.pi / 3) - sp.cos(sp.pi / 4))
assert exact_euclidean == 0
assert exact_spherical > 0
assert classify_symbol(4, 3, 4) == "euclidean-honeycomb"
assert classify_symbol(3, 3, 3) == "spherical-finite"
assert classify_symbol(3, 5, 3) == "hyperbolic-candidate"

class_to_code = {"local-invalid": 0, "hyperbolic-candidate": 1, "euclidean-honeycomb": 2, "spherical-finite": 3}
class_colors = ["#e5e7eb", "#fecaca", "#bbf7d0", "#bfdbfe"]
fig, axes = plt.subplots(1, 3, figsize=(11.2, 4.2), sharey=True)
for ax, q in zip(axes, [3, 4, 5]):
    grid = np.zeros((6, 6), dtype=int)
    for ip, p in enumerate(range(3, 9)):
        for ir, r in enumerate(range(3, 9)):
            grid[ip, ir] = class_to_code[classify_symbol(p, q, r)]
    ax.imshow(grid, origin="lower", cmap=ListedColormap(class_colors), vmin=0, vmax=3)
    ax.set_xticks(range(6), labels=range(3, 9))
    ax.set_yticks(range(6), labels=range(3, 9))
    ax.set_xlabel("r")
    ax.set_title(f"q = {q}")
axes[0].set_ylabel("p")
legend_handles = [Patch(facecolor=color, label=label) for label, color in zip(class_to_code.keys(), class_colors)]
fig.legend(handles=legend_handles, loc="lower center", ncol=4, frameon=False)
fig.suptitle("Admissibility classes for small Schlafli triples {p,q,r}")
fig.tight_layout(rect=[0, 0.12, 1, 0.93])
symbol_svg = FIG / "honeycomb-symbol-admissibility.svg"
fig.savefig(symbol_svg, format="svg")
plt.close(fig)
remember(symbol_svg, "figure", "Honeycomb symbol admissibility classes", "Matplotlib + SymPy SVG", "Locate the Euclidean boundary and the nearby finite/hyperbolic cases")

class_counts = symbol_df["classification"].value_counts().to_dict()
check_payload["invariants"]["honeycomb_symbols"] = {
    "exact_euclidean_margin_for_{4,3,4}": str(exact_euclidean),
    "classification_counts": {key: int(value) for key, value in class_counts.items()},
    "margin_{3,3,3}": float(symbol_margin(3, 3, 3)),
    "margin_{4,3,4}": float(symbol_margin(4, 3, 4)),
    "margin_{3,5,3}": float(symbol_margin(3, 5, 3)),
}

display_artifact(symbol_svg, width=850)
symbol_df[symbol_df["classification"].isin(["spherical-finite", "euclidean-honeycomb"])].head(12)


## 5. Close packing of equal 4-spheres: the D4 root shell

In four dimensions the `D4` lattice gives the densest equal-sphere packing. A compact way to see the local contact pattern is the `D4` root shell: all vectors with two nonzero coordinates, each equal to `+1` or `-1`. There are 24 such vectors. They all have norm `sqrt(2)`, so if sphere centers are separated by `sqrt(2)`, the touching radius is `sqrt(2)/2`. The fundamental volume of `D4` is 2, giving packing density `pi^2/16`.


In [ ]:
def d4_roots() -> np.ndarray:
    roots = []
    for i, j in combinations(range(4), 2):
        for si in [-1.0, 1.0]:
            for sj in [-1.0, 1.0]:
                v = np.zeros(4)
                v[i] = si
                v[j] = sj
                roots.append(v)
    return np.array(roots)


roots4 = d4_roots()
root_norms = np.linalg.norm(roots4, axis=1)
pairwise_root_distances = distance_matrix(roots4, roots4)
positive_distances = pairwise_root_distances[pairwise_root_distances > 1e-12]
radius = sqrt(2) / 2
d4_density = pi**2 / 16
assert roots4.shape == (24, 4)
assert np.allclose(root_norms, sqrt(2))
assert abs(radius - min(root_norms) / 2) < 1e-12
assert abs(d4_density - (pi**2 * radius**4 / 2) / 2) < 1e-12

root_df = pd.DataFrame(roots4, columns=["x1", "x2", "x3", "x4"])
root_df["norm"] = root_norms
roots_csv = DATA / "d4-root-neighbor-shell.csv"
write_csv(roots_csv, root_df.to_dict("records"))
remember(roots_csv, "data", "D4 root neighbor shell", "NumPy/Pandas CSV", "Verify the 24 equal-distance contacts around one 4-sphere")

basis = np.array([
    [1, 1, 1],
    [1, -1, 0],
    [1, 1, -2],
    [-3, 1, 1],
], dtype=float)
q_basis, _ = np.linalg.qr(basis)
roots3 = roots4 @ q_basis[:, :3]

fig = plt.figure(figsize=(9.2, 4.8))
ax = fig.add_subplot(121, projection="3d")
for point in roots3:
    ax.plot([0, point[0]], [0, point[1]], [0, point[2]], color="#94a3b8", linewidth=0.7, alpha=0.7)
ax.scatter(roots3[:, 0], roots3[:, 1], roots3[:, 2], c=roots4[:, 3], cmap="coolwarm", s=38, edgecolor="#111827", linewidth=0.3)
ax.scatter([0], [0], [0], color="#111827", s=38)
ax.set_title("24 projected D4 contact centers")
ax.set_xlabel("projection 1")
ax.set_ylabel("projection 2")
ax.set_zlabel("projection 3")
ax.view_init(elev=20, azim=-38)
ax.set_box_aspect((1, 1, 1))

ax2 = fig.add_subplot(122)
ax2.hist(root_norms, bins=[1.35, 1.45], color="#15803d", edgecolor="#14532d")
ax2.axvline(sqrt(2), color="#b91c1c", linewidth=2, label="sqrt(2)")
ax2.set_title("All 24 neighbors are at the same 4D distance")
ax2.set_xlabel("distance from central sphere center")
ax2.set_ylabel("count")
ax2.legend(frameon=False)
ax2.text(1.352, 18, f"kissing number = 24\nradius = sqrt(2)/2\ndensity = pi^2/16 = {d4_density:.4f}", fontsize=9, va="top")
fig.suptitle("Close packing in four dimensions: D4 local contact shell")
packing_svg = FIG / "d4-close-packing-neighbors.svg"
fig.savefig(packing_svg, format="svg")
plt.close(fig)
remember(packing_svg, "figure", "D4 close packing neighbor shell", "Matplotlib + SciPy distance SVG", "Count contacts and connect the visible shell to true 4D distances")

check_payload["invariants"]["d4_close_packing"] = {
    "kissing_number": int(len(roots4)),
    "minimum_center_distance": float(root_norms.min()),
    "sphere_radius": radius,
    "fundamental_volume": 2.0,
    "density": d4_density,
}

display_artifact(packing_svg, width=850)
check_payload["invariants"]["d4_close_packing"]


## 6. Statistical honeycombs: from one symbol to a distribution

A regular honeycomb is specified by a symbolic local rule. A statistical honeycomb is described by sampled cell data. The artifact below uses a planar Voronoi proxy because its cells can be inspected honestly on the page. The lesson is transferable: in higher dimensions the raw cell is harder to draw, but the shift from a single regular cell to distributions of volumes, neighbor counts, and nearest-neighbor scales is the same.


In [ ]:
rng = np.random.default_rng(20260511)
points = rng.random((90, 2))
vor = Voronoi(points)

finite_cells = []
for point_index, region_index in enumerate(vor.point_region):
    region = vor.regions[region_index]
    if not region or -1 in region:
        continue
    polygon = vor.vertices[region]
    if np.all((polygon >= 0.0) & (polygon <= 1.0)):
        x = polygon[:, 0]
        y = polygon[:, 1]
        area = 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))
        finite_cells.append({
            "point_index": point_index,
            "seed_x": points[point_index, 0],
            "seed_y": points[point_index, 1],
            "side_count": len(region),
            "area": area,
        })
cell_df = pd.DataFrame(finite_cells)
assert len(cell_df) >= 20
assert (cell_df["area"] > 0).all()
assert 4.0 <= cell_df["side_count"].mean() <= 8.0

cell_csv = DATA / "statistical-honeycomb-voronoi-cells.csv"
write_csv(cell_csv, cell_df.to_dict("records"))
remember(cell_csv, "data", "Statistical Voronoi honeycomb cells", "SciPy Voronoi + Pandas CSV", "Inspect side-count and area distributions for bounded sample cells")

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10.8, 4.8), gridspec_kw={"width_ratios": [1.15, 1]})
ax0.set_aspect("equal")
ax0.set_xlim(0, 1)
ax0.set_ylim(0, 1)
for simplex in vor.ridge_vertices:
    if -1 in simplex:
        continue
    segment = vor.vertices[simplex]
    if np.all((segment >= -0.05) & (segment <= 1.05)):
        ax0.plot(segment[:, 0], segment[:, 1], color="#475569", linewidth=0.7, alpha=0.8)
ax0.scatter(points[:, 0], points[:, 1], s=8, color="#b91c1c", alpha=0.65)
ax0.set_title("Voronoi honeycomb sample")
ax0.set_xticks([])
ax0.set_yticks([])

side_counts = sorted(cell_df["side_count"].unique())
ax1.hist(cell_df["side_count"], bins=np.arange(min(side_counts) - 0.5, max(side_counts) + 1.5), color="#2a6fbb", edgecolor="#1e3a8a")
ax1.axvline(cell_df["side_count"].mean(), color="#d97706", linewidth=2, label=f"mean = {cell_df['side_count'].mean():.2f}")
ax1.set_xlabel("bounded-cell side count")
ax1.set_ylabel("frequency")
ax1.set_title("Statistical replacement for a single symbol")
ax1.legend(frameon=False)
area_cv = float(cell_df["area"].std(ddof=1) / cell_df["area"].mean())
ax1.text(0.03, 0.95, f"bounded cells = {len(cell_df)}\narea CV = {area_cv:.2f}", transform=ax1.transAxes, va="top")
stat_svg = FIG / "statistical-honeycomb-voronoi.svg"
fig.savefig(stat_svg, format="svg")
plt.close(fig)
remember(stat_svg, "figure", "Statistical honeycomb Voronoi proxy", "SciPy Voronoi + Matplotlib SVG", "Compare sampled cell side counts with the idea of a regular symbol")

check_payload["invariants"]["statistical_honeycomb"] = {
    "sample_points": int(len(points)),
    "bounded_cells": int(len(cell_df)),
    "mean_side_count": float(cell_df["side_count"].mean()),
    "area_coefficient_of_variation": area_cv,
    "min_area": float(cell_df["area"].min()),
    "max_area": float(cell_df["area"].max()),
}

display_artifact(stat_svg, width=850)
cell_df.describe()


## Applied lab: move the slicing plane

A useful exploration should vary one geometric ingredient and protect one invariant. Here the ingredient is the fourth-coordinate level `w = t` in the 16-cell section. The combinatorics stay octahedral for `|t| < 1`, but the volume changes by the cubic law. Change the `lab_levels` array, rerun the cell, and check whether the plot and CSV still match the formula.


In [ ]:
lab_levels = np.linspace(-0.95, 0.95, 39)
lab_rows = []
for t in lab_levels:
    scale = 1.0 - abs(float(t))
    lab_rows.append({"w_level": float(t), "section_scale": scale, "normalized_volume": scale**3})
lab_df = pd.DataFrame(lab_rows)
assert np.isclose(lab_df.loc[lab_df["w_level"].abs().idxmin(), "normalized_volume"], 1.0)
assert lab_df["normalized_volume"].between(0, 1).all()

lab_csv = DATA / "section-volume-lab.csv"
write_csv(lab_csv, lab_df.to_dict("records"))
remember(lab_csv, "data", "Section-volume lab samples", "NumPy/Pandas CSV", "Vary the 4D slice level and verify cubic shrinkage")

fig, ax = plt.subplots(figsize=(7.6, 4.2))
ax.plot(lab_df["w_level"], lab_df["normalized_volume"], color="#6d28d9", linewidth=2)
ax.scatter(section_levels, [(1 - abs(t))**3 for t in section_levels], color="#d97706", zorder=3, label="displayed sections")
ax.set_xlabel("slice level w")
ax.set_ylabel("volume / volume at w = 0")
ax.set_title("Applied lab: the 16-cell section volume changes cubically")
ax.grid(True, color="#e5e7eb")
ax.legend(frameon=False)
lab_svg = FIG / "section-volume-lab.svg"
fig.savefig(lab_svg, format="svg")
plt.close(fig)
remember(lab_svg, "figure", "16-cell section-volume lab", "Matplotlib SVG", "Move the slicing plane and watch the normalized volume curve")
check_payload["invariants"]["section_volume_lab"] = {
    "sample_count": int(len(lab_df)),
    "max_normalized_volume": float(lab_df["normalized_volume"].max()),
    "min_normalized_volume": float(lab_df["normalized_volume"].min()),
}

display_artifact(lab_svg, width=760)
lab_df.head()


## Synthesis

Four-dimensional figures become teachable when every representation says what it forgets. The tesseract projection forgets metric scale but keeps edge incidence. The 16-cell section forgets the hidden coordinate but leaves a concrete octahedron whose volume records where the cut was made. Regular polytope tables reduce a large object to counts and duality, while the honeycomb-symbol test reduces admissibility to local cells plus a trigonometric boundary. Sphere packing uses an even more compressed invariant: one lattice shell, one kissing number, one density. Statistical honeycombs change the grammar again by replacing a perfect local symbol with sampled distributions.

The final cell writes the chapter check files and asserts that the figures, tables, data files, and core identities are present.

## Takeaways

- Four-dimensional geometry is best inspected through linked projections, sections, incidence tables, and invariant ledgers.
- Projection and slicing are not substitutes for the object; each is a controlled loss of information that should be named.
- Schlafli symbols, Euler-type counts, D4 packing data, and sampled honeycomb statistics give complementary checks on what a visual shadow is allowed to claim.


In [ ]:
# Final chapter-specific sanity checks and artifact manifest.
manifest_csv = TABLES / "artifact_manifest.csv"
remember(manifest_csv, "table", "Chapter artifact manifest", "CSV", "List every generated chapter-22 artifact and its inspection target")

visual_summary = {
    "chapter": CHAPTER_NO,
    "title": chapter["title"],
    "checks": {
        "printed_pages": chapter["printed"],
        "pdf_pages": chapter["pdf"],
        "figure_count": len(check_payload["figures"]),
        "html_count": len(check_payload["html"]),
        "table_count": len(check_payload["tables"]),
        "data_count": len(check_payload["data"]),
    },
    "figures": check_payload["figures"],
    "html": check_payload["html"],
    "tables": check_payload["tables"],
    "data": check_payload["data"],
    "invariants": check_payload["invariants"],
}
visual_summary_path = CHECKS / "visual_summary.json"
write_json(visual_summary_path, visual_summary)
remember(visual_summary_path, "check", "Visual summary and invariants", "JSON", "Audit source span, artifact paths, and chapter-specific checks")

invariant_path = CHECKS / "four-dimensional-geometry-invariants.json"
write_json(invariant_path, check_payload)
remember(invariant_path, "check", "Detailed 4D geometry invariants", "JSON", "Inspect all numerical and symbolic checks from the notebook")
write_csv(manifest_csv, artifact_records)

all_artifacts = [BOOK_ROOT / item["artifact"] for item in artifact_records]
assert_artifacts(all_artifacts, min_bytes=100)

assert len(vertices4) == 16 and len(tesseract_edges) == 32
assert check_payload["invariants"]["tesseract_projection"]["near_to_far_projected_edge_ratio"] > 1.0
assert all(abs(v) < 1e-12 for v in poly_df["euler_boundary"])
assert classify_symbol(4, 3, 4) == "euclidean-honeycomb"
assert str(exact_euclidean) == "0"
assert check_payload["invariants"]["d4_close_packing"]["kissing_number"] == 24
assert abs(check_payload["invariants"]["d4_close_packing"]["density"] - pi**2 / 16) < 1e-12
assert check_payload["invariants"]["statistical_honeycomb"]["bounded_cells"] >= 20
assert check_payload["invariants"]["statistical_honeycomb"]["min_area"] > 0
assert len(check_payload["figures"]) >= 6

final_sanity = {
    "artifact_count": len(all_artifacts),
    "figures": len(check_payload["figures"]),
    "html": len(check_payload["html"]),
    "tables": len(check_payload["tables"]),
    "data": len(check_payload["data"]),
    "checks": [rel(visual_summary_path), rel(invariant_path)],
    "artifact_sizes": {rel(path): path.stat().st_size for path in all_artifacts},
}
final_sanity
